# JEIT Revision — Reproducible Colab Run

This notebook runs the revised JEIT analysis package. It regenerates the matched model comparison, SHAP analysis, and reviewer-requested supplementary experiments. Upload the ZIP package created for this revision when prompted. The source dataset is included in the ZIP.

In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, shutil

uploaded = files.upload()  # Upload JEIT_Revised_Code_and_Results_Final.zip
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
workdir = Path('/content/jeit_revision')
if workdir.exists():
    shutil.rmtree(workdir)
workdir.mkdir(parents=True)
with ZipFile(zip_name) as zf:
    zf.extractall(workdir)
os.chdir(workdir)
print('Working directory:', Path.cwd())
print('Dataset exists:', Path('online_shoppers_intention.csv').exists())

In [ ]:
!python -m pip install -q -r requirements.txt

## Core matched comparison and TreeSHAP

This step regenerates the main matched comparison and the paired 1,200-session SHAP analysis. It can take several minutes.

In [ ]:
!python jeit_purchase_intention.py --data online_shoppers_intention.csv --steps main,shap

## Reviewer-requested supplementary experiments

This step regenerates matched metrics, repeated CV, nested CV, selector and subset-size sensitivity, paired bootstrap intervals, PageValues ablations, duplicate/group-aware analysis, and the baseline.

In [ ]:
!python reviewer_supplementary.py --data online_shoppers_intention.csv --out reviewer_results

## Bundle the generated outputs

The files below can be downloaded from Colab for comparison with the supplied final documents.

In [ ]:
from zipfile import ZipFile, ZIP_DEFLATED
from pathlib import Path
output_zip = Path('JEIT_Colab_Run_Outputs.zip')
with ZipFile(output_zip, 'w', ZIP_DEFLATED) as zf:
    for folder in ['results', 'reviewer_results']:
        for p in Path(folder).rglob('*'):
            if p.is_file():
                zf.write(p, p.as_posix())
print('Created:', output_zip, 'size:', output_zip.stat().st_size)
files.download(str(output_zip))